# 03 — Group Duplicates, Assemble Final Dataset, Train/Val Split

`02_preprocessing.ipynb` stays untouched as pure crop/pad/resize/cache. This
notebook handles everything downstream of that.

Notebook 01 already did the detection work and left two artifacts on disk:

- `data/train_groups.json` — near-duplicate clusters **within** `Training`
  (path -> group id), built via perceptual hashing at the manually-validated
  threshold of 4.
- `data/clean_test.json` — the list of `Testing` images that have **no**
  near-duplicate anywhere in `Training`, i.e. the ones that are actually safe
  to evaluate on.

This notebook:

1. Keeps every Training image, but uses `train_groups.json` to make sure an
   entire duplicate cluster lands on the same side of the train/val split —
   never split across both. Duplicates are not deleted here.
2. Drops Testing images that leak into Training (via `clean_test.json`) —
   this one **is** a removal, since a leaked test image can't give a fair
   evaluation no matter how training is grouped.
3. Runs `GroupShuffleSplit` on Training, grouped by cluster.
4. Assembles the final `X`/`y` arrays from the preprocessed cache built in
   `02_preprocessing.ipynb`.

In [5]:
import csv, json, random
from pathlib import Path
from collections import defaultdict

import numpy as np
from sklearn.model_selection import GroupShuffleSplit

random.seed(42)
np.random.seed(42)

with open("data/manifest.csv") as f:
    rows = list(csv.DictReader(f))

records = [(r["split"], r["class"], Path(r["path"])) for r in rows]
CLASSES = sorted({r["class"] for r in rows})

print(f"{len(records)} total records, classes: {CLASSES}")

7200 total records, classes: ['glioma', 'meningioma', 'notumor', 'pituitary']


## 1. Rebuild the cache lookup

`02_preprocessing.ipynb` writes cropped/padded/resized arrays to
`data/processed/<split>/<class>/<stem>.npy`, but the in-memory `cache` dict
that mapped original path -> npy path lives only in that notebook's kernel.
Rebuilding the mapping from disk here means this notebook doesn't depend on
that kernel still being alive.

In [6]:
PROCESSED = Path("data/processed")

cache = {}
missing = []
for split, cls, p in records:
    npy_path = PROCESSED / split / cls / f"{p.stem}.npy"
    if npy_path.exists():
        cache[p] = npy_path
    else:
        missing.append(p)

print(f"Found cached arrays for {len(cache)} / {len(records)} records")
if missing:
    print(f"WARNING: {len(missing)} records have no cached .npy — "
          f"run 02_preprocessing.ipynb's caching cell first.")
    print("First few missing:", missing[:3])

Found cached arrays for 7200 / 7200 records


## 2. Load duplicate groups for Training

No removal here — every Training image is kept. `train_groups.json` just
tells us which images belong to the same near-duplicate cluster, so the split
step below can keep clusters intact.

In [7]:
with open("data/train_groups.json") as f:
    train_groups = json.load(f)          # path(str) -> group id

train_records = [(s, c, p) for s, c, p in records if s == "Training"]

group_sizes = defaultdict(int)
for p_str in train_groups.values():
    group_sizes[p_str] += 1
sizes = sorted(group_sizes.values(), reverse=True)

print(f"{len(train_records)} training images -> {len(set(train_groups.values()))} groups")
print(f"Groups of size 1 (unique):     {sum(1 for s in sizes if s == 1)}")
print(f"Groups of size 2-5:            {sum(1 for s in sizes if 2 <= s <= 5)}")
print(f"Groups of size >5:             {sum(1 for s in sizes if s > 5)}")
print(f"Largest group size:            {sizes[0] if sizes else 0}")

5600 training images -> 3989 groups
Groups of size 1 (unique):     3201
Groups of size 2-5:            719
Groups of size >5:             69
Largest group size:            22


## 3. Clean Testing

This one is a real removal, not grouping — a `Testing` image with a
near-duplicate in `Training` gives an inflated, meaningless accuracy no
matter how `Training` itself is split, so it comes out.

In [8]:
with open("data/clean_test.json") as f:
    clean_test_paths = set(json.load(f))  # test paths with NO train duplicate

test_records_all = [(s, c, p) for s, c, p in records if s == "Testing"]
test_records_clean = [(s, c, p) for s, c, p in test_records_all if str(p) in clean_test_paths]

print(f"Testing:  {len(test_records_all)} -> {len(test_records_clean)} after removing leaked images "
      f"({len(test_records_all) - len(test_records_clean)} leaked images removed)")

Testing:  1600 -> 886 after removing leaked images (714 leaked images removed)


**Not covered by either file above:** duplicates purely *within* Testing
(two near-identical images both in the test set). Notebook 01 only checked
train-vs-test leakage and within-Training clusters, not within-Testing
redundancy. Flag it if you want that check added too.

## 4. GroupShuffleSplit

`test_size=0.15` -> ~15% of Training held out as validation, grouped by
cluster so no group crosses the boundary. `random_state=42` to match the rest
of the project.

In [9]:
paths_arr = [str(p) for s, c, p in train_records]
classes_arr = [c for s, c, p in train_records]
groups_arr = [train_groups[str(p)] for s, c, p in train_records]

gss = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
train_idx, val_idx = next(gss.split(paths_arr, classes_arr, groups=groups_arr))

print(f"train: {len(train_idx)}   val: {len(val_idx)}")

train: 4803   val: 797


## 5. Verify no group crosses the split

Should always print 0 — if it doesn't, something's wrong with how `groups=`
was passed above, not with the data itself.

In [10]:
train_groups_set = {groups_arr[i] for i in train_idx}
val_groups_set = {groups_arr[i] for i in val_idx}
overlap = train_groups_set & val_groups_set
print(f"Groups appearing on both sides: {len(overlap)}")
assert len(overlap) == 0, "Group leakage across train/val split!"

Groups appearing on both sides: 0


## 6. Class balance check

`GroupShuffleSplit` does **not** stratify by class, only by group — grouping
can skew balance a little, worth eyeballing rather than assuming it held.

In [11]:
def class_counts_idx(idxs):
    counts = defaultdict(int)
    for i in idxs:
        counts[classes_arr[i]] += 1
    total = len(idxs)
    return {c: (n, round(100 * n / total, 1)) for c, n in sorted(counts.items())}

print("train:")
for c, (n, pct) in class_counts_idx(train_idx).items():
    print(f"  {c:<14} {n:>5}  ({pct}%)")

print("val:")
for c, (n, pct) in class_counts_idx(val_idx).items():
    print(f"  {c:<14} {n:>5}  ({pct}%)")

train:
  glioma          1174  (24.4%)
  meningioma      1183  (24.6%)
  notumor         1246  (25.9%)
  pituitary       1200  (25.0%)
val:
  glioma           226  (28.4%)
  meningioma       217  (27.2%)
  notumor          154  (19.3%)
  pituitary        200  (25.1%)


If any class is off by more than a few percentage points, that's worth
knowing before training — but don't force-stratify on top of grouping unless
the skew is severe, since that risks re-splitting a cluster across train/val.

## 7. Assemble final arrays

Pulls the cached `.npy` arrays (cropped/padded/resized by
`02_preprocessing.ipynb`) for the final train/val/test record lists, stacks
them, and scales to `[0, 1]`. Duplicates are still present in `X_train` here
— they were grouped, not removed.

In [12]:
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}

def build_arrays_from_records(recs):
    X = np.stack([np.load(cache[p]) for s, c, p in recs]).astype(np.float32) / 255.0
    y = np.array([CLASS_TO_IDX[c] for s, c, p in recs], dtype=np.int64)
    return X, y

train_final = [train_records[i] for i in train_idx]
val_final = [train_records[i] for i in val_idx]

X_train, y_train = build_arrays_from_records(train_final)
X_val, y_val = build_arrays_from_records(val_final)
X_test, y_test = build_arrays_from_records(test_records_clean)

print(f"X_train: {X_train.shape}   y_train: {y_train.shape}")
print(f"X_val  : {X_val.shape}   y_val  : {y_val.shape}")
print(f"X_test : {X_test.shape}   y_test : {y_test.shape}")

X_train: (4803, 224, 224)   y_train: (4803,)
X_val  : (797, 224, 224)   y_val  : (797,)
X_test : (886, 224, 224)   y_test : (886,)


In [13]:
np.savez_compressed(
    "data/final_dataset.npz",
    X_train=X_train, y_train=y_train,
    X_val=X_val, y_val=y_val,
    X_test=X_test, y_test=y_test,
    classes=np.array(CLASSES),
)
print("Saved data/final_dataset.npz")
print("\nClass index mapping:")
for c, i in CLASS_TO_IDX.items():
    print(f"  {i}: {c}")

Saved data/final_dataset.npz

Class index mapping:
  0: glioma
  1: meningioma
  2: notumor
  3: pituitary


## Next

Model notebook loads `data/final_dataset.npz` directly — `X_train`/`y_train`
for training, `X_val`/`y_val` for tuning and early stopping, `X_test`/`y_test`
touched exactly once, at the end.